# Comparison Summary · Methods and Results Side by Side

This notebook does two things:
1. Runs a **controlled matrix** — the same two anchor questions across all four retrieval modes on a single rich index — so differences are attributable to the retrieval method.
2. Loads every **recorded lab run** (from `notebooks/results/lab_runs.jsonl`) into one comparison table.

**Anchor questions**
- **Q1 (precise / keyword):** site-investigation objectives — favours full-text.
- **Q2 (conceptual / multi-hop):** groundwater + clay risks and design — favours vector / hybrid / agentic.

## Step 1 — Bootstrap

In [1]:
import sys
from pathlib import Path

NB_DIR = Path.cwd()
sys.path.insert(0, str(NB_DIR if (NB_DIR / 'lab_runtime.py').exists() else NB_DIR / 'notebooks'))
import lab_runtime as lab

info = lab.bootstrap()
info

{'repo_root': 'C:\\Users\\samsonlee\\rag-on-azure-lab',
 'env_values_loaded': 89,
 'search_endpoint': 'https://aisearchlabl4qxbh.search.windows.net',
 'search_configured': True,
 'embedding_deployment': 'text-embedding-3-large-vector',
 'chat_deployment': 'gpt-5-4-mini-chat',
 'agentic_planning_model': 'gpt-5-4-mini-search',
 'canonical_index': 'ai-search-lab-index'}

## Step 2 — Use the richest index for a fair cross-mode comparison

In [2]:
job = lab.ingest(skill_profile='visual_nlp', reuse=True)
lab.chunk_overview(job)

Reusing existing 'visual_nlp' ingestion (doc_id=edbcb680, 414 chunks). Pass reuse=False to force a fresh run.


{'doc_id': 'edbcb680033d4886af1b6cfab59ccddc',
 'skill_profile': 'visual_nlp',
 'chunk_count': 414,
 'avg_tokens': 208.4,
 'max_tokens': 1976,
 'chunks_with_summary': 414,
 'chunks_with_keyword_hints': 414,
 'chunks_with_image_description': 0}

## Step 3 — Controlled matrix: 2 questions × 4 retrieval modes

Each cell is a full grounded chat turn (retrieval + synthesis), the same as the UI.

In [3]:
Q1 = 'What are the objectives of site investigation for ELS works?'
Q2 = 'Given a site with high groundwater and clay layers, what are the key excavation risks and design considerations?'

import pandas as pd

matrix = []
for qlabel, q in [('Q1 precise', Q1), ('Q2 multi-hop', Q2)]:
    for mode in ['full_text', 'vector', 'hybrid', 'agentic']:
        resp = lab.ask(q, job=job, retrieval_mode=mode, record_as=f'compare_{mode}_{qlabel.split()[0]}')
        diag = resp.diagnostics or {}
        top = resp.citations[0] if resp.citations else None
        matrix.append({
            'question': qlabel,
            'mode': mode,
            'citations': len(resp.citations),
            'answer_chars': len(resp.answer),
            'synthesis': diag.get('answer_synthesis_enabled'),
            'top_section': (' > '.join(top.section_path) if top and top.section_path else (top.title if top else '—')),
        })
df = pd.DataFrame(matrix)
df

,question,mode,citations,answer_chars,synthesis,top_section
0,Q1 precise,full_text,8,1080,False,2.1 Site Investigation
1,Q1 precise,vector,8,1063,False,2.1 Site Investigation
2,Q1 precise,hybrid,7,1001,False,2.1 Site Investigation
3,Q1 precise,agentic,8,888,True,2.1 Site Investigation
4,Q2 multi-hop,full_text,8,1005,False,10.2.3 Groundwater Monitoring
5,Q2 multi-hop,vector,8,987,False,5.2 Site Conditions Particularly Vulnerable to...
6,Q2 multi-hop,hybrid,8,1041,False,7.2 Overall Stability
7,Q2 multi-hop,agentic,8,3214,True,7.2 Overall Stability


### Read the matrix

Expect full-text to win on Q1's exact terminology, vector/hybrid to broaden Q2 recall, and agentic to return the most cross-section citations on Q2.

## Step 4 — Answers side by side (Q2)

The qualitative difference is clearest on the multi-hop question.

In [4]:
for mode in ['full_text', 'vector', 'hybrid', 'agentic']:
    print('=' * 80)
    print('MODE:', mode)
    resp = lab.ask(Q2, job=job, retrieval_mode=mode)
    print(resp.answer.strip()[:700])
    print()

MODE: full_text


However, their locations, length and positioning of response zones should be properly planned with due consideration given to the hydrogeological condition of the groundwater regime adjacent to the site. Where the ground conditions comprise stratified marine clay underlain by sandy soils (e.g. [5]

Supporting evidence:
- Given the temporary nature of ELS works, DGWL should be related to possible scenarios that could occur within the duration of the works for different limit states. It is not necessary to consider the effects of long-term and extreme events (e.g. due to climate change). Based on local experience and practice, the following considerations are usually adopted i [6]
- The design

MODE: vector


The presence of loose fill renders a site more susceptible to excessive ground loss during piling operations and bulk excavation, particularly where the groundwater table is near the ground surface. This is commonly the case in reclaimed land where the fill was placed by the hydraulic filling method. [2]

Supporting evidence:
- ULS Design (0.5 m of groundwater (a) Inspect and examine the performance of the ELS works and the response of nearby sensitive receivers with respect to their structural stability and serviceability. level below the highest DGWL) (b) Investigate causes and any correlations with observed changes in groundwater levels. [4]
- Site-specific groundwater and drainage condit

MODE: hybrid


GEO Publication No. 1/2023, **Deep Excavation Design and Construction**, is a Hong Kong Geotechnical Engineering Office publication first published in December 2023. Loss of overall stability is likely to occur in excavations near a steeply-sloping site with a high groundwater table, or where a weak subsoil layer (e.g. loose sand or soft clay) is. [5]

Supporting evidence:
- In order to minimise disturbance to the adjacent ground due to the boring operation, the pressure of the compressed air should be carefully assessed and monitored, especially for sites with a high groundwater table and thick layers of loose fill, bouldery colluvium or rockfill. Trial boring is usually conducted and shoul

MODE: agentic


- **Key excavation risks** on a site with **high groundwater and clay layers** include: - **Groundwater ingress / seepage** into the excavation, which can raise wall forces and cause instability. [1][2] - **Uplift / base heave**, especially where a **low-permeability layer overlies sandy soil** and **artesian pressure** exceeds overburden pressure. [3][4] - **Piping / sinkholes** where upward seepage reduces effective stress and washes soil out, particularly in **loose sandy layers** with high permeability. [5] - **Overall stability loss**, especially near **steep slopes**, where groundwater is high, or where weak layers such as **soft clay** exist below the wall toe. [6][7] - **Settlement o



## Step 5 — Every recorded lab run

This pulls the runs each lab notebook recorded, so you can compare ingestion profiles and modes across the whole workshop.

In [5]:
import pandas as pd

runs = lab.load_runs()
print('recorded runs:', len(runs))
if runs:
    rdf = pd.DataFrame([
        {
            'label': r['label'],
            'skill_profile': r['skill_profile'],
            'retrieval_mode': r['retrieval_mode'],
            'scoring_profile': r['scoring_profile'],
            'citations': r['citation_count'],
            'elapsed_s': r['elapsed_s'],
            'answer_preview': r['answer'][:90].replace(chr(10), ' '),
        }
        for r in runs
    ])
    display(rdf)

recorded runs: 19


,label,skill_profile,retrieval_mode,scoring_profile,citations,elapsed_s,answer_preview
0,lab03_baseline_fulltext,baseline_extract,full_text,default,8,1.98,The main objectives of site investigation are ...
1,lab04_chunkvector_full_text,chunk_vector,full_text,default,8,1.65,"However, their locations, length and positioni..."
2,lab04_chunkvector_vector,chunk_vector,vector,default,8,7.37,The presence of loose fill renders a site more...
3,lab04_chunkvector_hybrid,chunk_vector,hybrid,default,8,4.24,Loss of overall stability is likely to occur i...
4,lab05_genai_hybrid,genai_enrichment,hybrid,default,8,7.52,"GEO Publication No. 1/2023, **Deep Excavation ..."
5,lab05_genai_agentic,genai_enrichment,agentic,default,8,8.78,- **Key excavation risks** in high groundwater...
6,lab06_visual_hybrid,visual_nlp,hybrid,default,7,7.30,"However, this method only calculates the later..."
7,lab06_visual_agentic,visual_nlp,agentic,default,8,17.71,- The figures show that **ground settlement an...
8,lab07_agentic_q2,genai_enrichment,agentic,default,8,8.69,Key excavation risks and design considerations...
9,lab07_hybrid_q2,genai_enrichment,hybrid,default,8,7.50,"GEO Publication No. 1/2023, **Deep Excavation ..."


## How to choose a retrieval mode

| Mode | Best for | Why |
| --- | --- | --- |
| **Full-text** | exact terminology, definitions, standards, tables | lexical BM25 matches explicit wording |
| **Vector** | paraphrased / conceptual questions | embedding similarity finds meaning, not words |
| **Hybrid** | precise + contextual answers | fuses keyword precision with semantic recall |
| **Agentic** | multi-hop reasoning, synthesis across sections | plans subqueries and grounds a cited answer |

And across ingestion profiles: each enrichment layer (chunking → vectors → generative cues → visual/NLP) adds retrievable signal, with the biggest gains showing up on conceptual and diagram-grounded questions.